# Full two-dataset dimensionality reduction benchmark

This notebook runs **PCA, kernel PCA, SNE, t-SNE, PCA + t-SNE, UMAP, MDS,
PCoA, Isomap, LLE, and a new PCA + UMAP hybrid** on HAM10000 and Blood Cell
Anomaly Detection 2025. It contains preprocessing, eleven algorithms, two settings per algorithm,
held-out classification, metrics, plots, timing, and exploratory tests in
the notebook itself. No repository attachment or Fashion-MNIST data is needed.

Attach the public Kaggle datasets `kmader/skin-cancer-mnist-ham10000` and
`alitaqishah/blood-cell-anomaly-detection-2025` before running all cells.
This is a computational pilot, not a clinical diagnostic evaluation.

## 1. Environment and input files

The default run uses three shared splits, 750 training records, and 300 test
records per split. Set the `AML_NOTEBOOK_*` environment variables for a
smaller local check. Missing dependencies are installed only when needed.

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for module, package in [("umap", "umap-learn>=0.5.9"),
                        ("statsmodels", "statsmodels>=0.14"),
                        ("threadpoolctl", "threadpoolctl>=3.5")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ.setdefault(name, "1")
default_root = (Path("/kaggle/working/aml-benchmark") if Path("/kaggle/working").exists()
                else Path.cwd() / "notebook_output")
ROOT = Path(os.getenv("AML_NOTEBOOK_OUTPUT", str(default_root))).resolve()
DATA, RESULTS = ROOT / "data", ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["NUMBA_CACHE_DIR"] = str(ROOT / "numba_cache")
SEEDS = int(os.getenv("AML_NOTEBOOK_SEEDS", "3"))
TRAIN_N = int(os.getenv("AML_NOTEBOOK_TRAIN_N", "750"))
TEST_N = int(os.getenv("AML_NOTEBOOK_TEST_N", "300"))
assert SEEDS >= 1 and TRAIN_N >= 100 and TEST_N >= 40
print(f"Output: {ROOT}; seeds={SEEDS}; training={TRAIN_N}; test={TEST_N}")

In [ ]:
import hashlib, shutil
import pandas as pd
from IPython.display import Image, display

INPUT_OVERRIDES = {"HAM10000_metadata.csv": None,
                   "hmnist_8_8_RGB.csv": None,
                   "blood_cell_anomaly_detection.csv": None}
targets = {"HAM10000_metadata.csv": DATA / "ham/HAM10000_metadata.csv",
           "hmnist_8_8_RGB.csv": DATA / "ham/hmnist_8_8_RGB.csv",
           "blood_cell_anomaly_detection.csv": DATA / "blood/blood_cell_anomaly_detection.csv"}
def find_input(filename):
    if INPUT_OVERRIDES[filename]:
        path = Path(INPUT_OVERRIDES[filename])
        if not path.is_file(): raise FileNotFoundError(path)
        return path
    locations = []
    for root in (Path("/kaggle/input"), Path.cwd() / "data"):
        if root.exists(): locations += list(root.rglob(filename))
    locations = list({str(p.resolve()): p for p in locations}.values())
    if len(locations) != 1:
        raise RuntimeError(f"Expected one {filename}, found {locations}. Set INPUT_OVERRIDES.")
    return locations[0]
manifest = []
for filename, target in targets.items():
    source = find_input(filename)
    target.parent.mkdir(parents=True, exist_ok=True)
    if source.resolve() != target.resolve(): shutil.copy2(source, target)
    manifest.append({"file": filename, "bytes": target.stat().st_size,
                     "sha256": hashlib.sha256(target.read_bytes()).hexdigest()})
display(pd.DataFrame(manifest))

## 2. Dataset checks and feature selection

HAM10000 uses 192 values from the 8 × 8 RGB pixel CSV. We check exact
pixel/metadata label alignment and split by `lesion_id`. The positive class
is melanoma. Blood cells use 22 measured morphology, color, and routine
blood-count variables. We exclude cell type, disease category, and supplied
anomaly/confidence scores that could reveal the answer. The blood-cell file
has no patient ID; row splits cannot prove patient-level generalization.

In [ ]:
import json, time
import numpy as np
import matplotlib.pyplot as plt
import umap
from scipy.optimize import minimize
from scipy import stats
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE, MDS, Isomap, LocallyLinearEmbedding
from sklearn.metrics import pairwise_distances, confusion_matrix, log_loss, roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from statsmodels.stats.anova import AnovaRM
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multitest import multipletests

METHODS = ["PCA", "Kernel PCA", "SNE", "t-SNE", "PCA + t-SNE", "UMAP", "MDS", "PCoA", "Isomap"]
GRIDS = {
    "PCA": [{"whiten": False}, {"whiten": True}],
    "Kernel PCA": [{"gamma_factor": 0.5}, {"gamma_factor": 2.0}],
    "SNE": [{"perplexity": 15}, {"perplexity": 30}],
    "t-SNE": [{"perplexity": 15}, {"perplexity": 30}],
    "PCA + t-SNE": [{"perplexity": 15}, {"perplexity": 30}],
    "UMAP": [{"n_neighbors": 15}, {"n_neighbors": 30}],
    "MDS": [{"max_iter": 80}, {"max_iter": 150}],
    "PCoA": [{"distance": "cosine"}, {"distance": "manhattan"}],
    "Isomap": [{"n_neighbors": 10}, {"n_neighbors": 25}],
}

In [ ]:
def load_dataset(name: str):
    if name == "HAM10000":
        pixels = pd.read_csv(DATA / "ham" / "hmnist_8_8_RGB.csv")
        meta = pd.read_csv(DATA / "ham" / "HAM10000_metadata.csv")
        mapping = {"akiec": 0, "bcc": 1, "bkl": 2, "df": 3,
                   "nv": 4, "vasc": 5, "mel": 6}
        if len(pixels) != len(meta) or not np.array_equal(
            pixels["label"].to_numpy(), meta["dx"].map(mapping).to_numpy()
        ):
            raise ValueError("HAM pixel/metadata row alignment failed")
        x = pixels.drop(columns="label").to_numpy(dtype=np.float32) / 255.0
        y = (meta["dx"] == "mel").astype(int).to_numpy()
        groups = meta["lesion_id"].astype(str).to_numpy()
        return x, y, groups, list(pixels.columns[:-1])
    if name == "Blood cell":
        df = pd.read_csv(DATA / "blood" / "blood_cell_anomaly_detection.csv")
        # Restrict to measured cell morphology, colour, and routine blood counts.
        # Exclude cell_type, disease_category, and three model/labeller scores.
        features = ["cell_diameter_um", "nucleus_area_pct", "chromatin_density",
                    "cytoplasm_ratio", "circularity", "eccentricity",
                    "granularity_score", "lobularity_score", "membrane_smoothness",
                    "cell_area_px", "perimeter_px", "mean_r", "mean_g", "mean_b",
                    "stain_intensity", "wbc_count_per_ul",
                    "rbc_count_millions_per_ul", "hemoglobin_g_dl",
                    "hematocrit_pct", "platelet_count_per_ul", "mcv_fl", "mchc_g_dl"]
        if df["cell_id"].duplicated().any() or df[features].isna().any().any():
            raise ValueError("Unexpected duplicate ID or missing blood feature")
        return df[features].to_numpy(dtype=np.float32), df["anomaly_label"].to_numpy(dtype=int), None, features
    raise ValueError(name)

In [ ]:
profiles = []
for dataset in ("HAM10000", "Blood cell"):
    x, y, groups, features = load_dataset(dataset)
    profiles.append({"dataset": dataset, "rows": len(y), "features": len(features),
                     "positive_n": int(y.sum()), "positive_rate": float(y.mean()),
                     "missing_features": int(np.isnan(x).sum()),
                     "group_count": len(np.unique(groups)) if groups is not None else "unavailable"})
profile = pd.DataFrame(profiles)
display(profile)
ax = profile.plot.bar(x="dataset", y="positive_rate", legend=False,
                      color=["#23668A", "#D17A42"], figsize=(7, 3.5))
ax.set(ylabel="Positive-class share", ylim=(0, 1), title="Class prevalence")
plt.tight_layout(); plt.show()

## 3. Shared preprocessing and algorithm settings

Every method sees the same held-out records within a dataset and seed.
Standardization is fit on training records only. Setting A is the primary
comparison; setting B measures parameter sensitivity. Selecting the better
setting using the test results would bias performance upward.

In [ ]:
def balanced_subsample(indices, labels, n, seed):
    if len(indices) <= n:
        return indices
    chosen, _ = train_test_split(indices, train_size=n, stratify=labels[indices], random_state=seed)
    return np.sort(chosen)

def split_data(x, y, groups, seed, train_n, test_n):
    idx = np.arange(len(y))
    if groups is None:
        tr, te = train_test_split(idx, test_size=0.28, stratify=y, random_state=seed)
    else:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.28, random_state=seed)
        tr, te = next(splitter.split(x, y, groups))
    tr = balanced_subsample(tr, y, train_n, seed)
    te = balanced_subsample(te, y, test_n, seed + 100)
    if groups is not None and set(groups[tr]) & set(groups[te]):
        raise AssertionError("Lesion leakage")
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        raise ValueError("Split has only one class")
    scaler = StandardScaler().fit(x[tr])
    return scaler.transform(x[tr]), scaler.transform(x[te]), y[tr], y[te], tr, te

## 4. Embedding and held-out transformation

PCA preserves variance; kernel PCA uses nonlinear RBF similarity. MDS
preserves Euclidean distances, PCoA decomposes a centered distance matrix,
and Isomap uses neighbor-graph geodesics. SNE, t-SNE, and UMAP emphasize
local neighborhoods. PCA + t-SNE filters features before t-SNE.

PCA, kernel PCA, UMAP, and Isomap transform unseen records directly. PCoA
uses a Gower extension. SNE, t-SNE, and MDS use a documented five-neighbor
approximation because these implementations lack a test transform. Their
measured scores include this approximation.

In [ ]:
def sne_embedding(x, perplexity, seed, steps=130):
    """Symmetric Gaussian SNE, with Gaussian low-dimensional affinity."""
    from sklearn.metrics import pairwise_distances
    distance = pairwise_distances(x, metric="sqeuclidean")
    n = len(x)
    cond = np.zeros((n, n), dtype=np.float32)
    target_entropy = np.log(perplexity)
    for i in range(n):
        lo, hi = 1e-5, 1e3
        d = np.delete(distance[i], i)
        for _ in range(24):
            beta = (lo + hi) / 2
            p = np.exp(-beta * (d - d.min()))
            p /= p.sum()
            entropy = -(p * np.log(p + 1e-12)).sum()
            if entropy > target_entropy:
                lo = beta
            else:
                hi = beta
        cond[i, np.arange(n) != i] = p
    p_sym = (cond + cond.T) / (2 * n)
    np.fill_diagonal(p_sym, 0)
    y0 = PCA(n_components=2, random_state=seed).fit_transform(x).astype(np.float64)
    y0 = 0.1 * y0 / max(float(np.std(y0)), 1e-12)
    trace = []
    iterations = [0]
    def objective(flat):
        y = flat.reshape(n, 2)
        d = pairwise_distances(y, metric="sqeuclidean")
        q = np.exp(-d / 2)
        np.fill_diagonal(q, 0)
        q /= q.sum()
        loss = np.sum(p_sym * (np.log(p_sym + 1e-12) - np.log(q + 1e-12)))
        a = p_sym - q
        grad = 2 * (a.sum(axis=1, keepdims=True) * y - a @ y)
        return float(loss), grad.ravel()
    def callback(intermediate):
        iterations[0] += 1
        if iterations[0] % 10 == 0:
            trace.append([iterations[0], objective(intermediate)[0]])
    result = minimize(objective, y0.ravel(), jac=True, method="L-BFGS-B",
                      callback=callback, options={"maxiter": steps})
    trace.append([int(result.nit), float(result.fun)])
    return result.x.reshape(n, 2), trace

In [ ]:
def pcoa_transform(x_train, x_test, metric):
    """Classical principal coordinates with Gower out-of-sample extension."""
    d = pairwise_distances(x_train, metric=metric)
    d2 = d ** 2
    row_mean = d2.mean(axis=1)
    grand_mean = d2.mean()
    gram = -0.5 * (d2 - row_mean[:, None] - row_mean[None, :] + grand_mean)
    eigenvalues, eigenvectors = np.linalg.eigh(gram)
    order = np.argsort(eigenvalues)[::-1][:2]
    lam = eigenvalues[order]
    vectors = eigenvectors[:, order]
    scale = np.sqrt(np.maximum(lam, 0))
    inverse_scale = np.divide(1.0, scale, out=np.zeros_like(scale), where=scale > 1e-10)
    z_train = vectors * scale
    test_d2 = pairwise_distances(x_test, x_train, metric=metric) ** 2
    centered = -0.5 * (test_d2 - row_mean[None, :] - test_d2.mean(axis=1, keepdims=True) + grand_mean)
    z_test = centered @ (vectors * inverse_scale)
    return z_train, z_test

def embed(method, params, x_train, x_test, seed):
    trace = None
    if method == "PCA":
        model = PCA(n_components=2, whiten=params["whiten"], random_state=seed)
        return model.fit_transform(x_train), model.transform(x_test), trace
    if method == "Kernel PCA":
        model = KernelPCA(n_components=2, kernel="rbf",
                          gamma=params["gamma_factor"] / x_train.shape[1],
                          eigen_solver="arpack", random_state=seed)
        return model.fit_transform(x_train), model.transform(x_test), trace
    if method == "UMAP":
        model = umap.UMAP(n_components=2, n_neighbors=params["n_neighbors"],
                          min_dist=0.1, random_state=seed, n_jobs=1)
        return model.fit_transform(x_train), model.transform(x_test), trace
    if method == "PCoA":
        z_train, z_test = pcoa_transform(x_train, x_test, params["distance"])
        return z_train, z_test, trace
    if method == "Isomap":
        model = Isomap(n_components=2, n_neighbors=params["n_neighbors"], n_jobs=1)
        return model.fit_transform(x_train), model.transform(x_test), trace
    if method == "MDS":
        model = MDS(n_components=2, metric=True, n_init=1, max_iter=params["max_iter"],
                    normalized_stress="auto", random_state=seed, n_jobs=1)
        z_train = model.fit_transform(x_train)
        mapper = KNeighborsRegressor(n_neighbors=5, weights="distance", n_jobs=1)
        mapper.fit(x_train, z_train)
        return z_train, mapper.predict(x_test), [[int(model.n_iter_), float(model.stress_)]]
    source_train, source_test = x_train, x_test
    if method == "PCA + t-SNE":
        pre = PCA(n_components=min(30, x_train.shape[1] - 1), random_state=seed)
        source_train = pre.fit_transform(x_train)
        source_test = pre.transform(x_test)
    if method == "SNE":
        z_train, trace = sne_embedding(source_train, params["perplexity"], seed)
    else:
        model = TSNE(n_components=2, perplexity=params["perplexity"],
                     init="pca", learning_rate="auto", max_iter=500,
                     method="barnes_hut", random_state=seed)
        z_train = model.fit_transform(source_train)
        trace = [[500, float(model.kl_divergence_)]]
    # Explicit approximation because sklearn t-SNE and this SNE have no transform.
    mapper = KNeighborsRegressor(n_neighbors=5, weights="distance", n_jobs=1)
    mapper.fit(source_train, z_train)
    return z_train, mapper.predict(source_test), trace

### Notebook extensions: LLE and PCA + UMAP

Locally Linear Embedding (LLE) reconstructs each point from nearby points
and seeks low-dimensional coordinates preserving those reconstruction
weights. The new PCA + UMAP hybrid removes low-variance input directions
before learning UMAP's neighborhood graph. Both are exploratory additions;
neither is guaranteed to improve held-out performance. PCA + t-SNE remains
in the main method list and is evaluated alongside them.

In [ ]:
METHODS += ["LLE", "PCA + UMAP"]
GRIDS["LLE"] = [{"n_neighbors": 15}, {"n_neighbors": 30}]
GRIDS["PCA + UMAP"] = [{"n_neighbors": 15}, {"n_neighbors": 30}]
base_embed = embed
def embed(method, params, x_train, x_test, seed):
    if method == "LLE":
        model = LocallyLinearEmbedding(n_components=2,
                                       n_neighbors=params["n_neighbors"],
                                       method="standard", eigen_solver="arpack",
                                       random_state=seed, n_jobs=1)
        return model.fit_transform(x_train), model.transform(x_test), None
    if method == "PCA + UMAP":
        pre = PCA(n_components=min(30, x_train.shape[1] - 1), random_state=seed)
        xtr, xte = pre.fit_transform(x_train), pre.transform(x_test)
        model = umap.UMAP(n_components=2, n_neighbors=params["n_neighbors"],
                          min_dist=0.1, random_state=seed, n_jobs=1)
        return model.fit_transform(xtr), model.transform(xte), None
    return base_embed(method, params, x_train, x_test, seed)
print("Methods:", METHODS)

In [ ]:
display(pd.DataFrame([{'method': m, 'setting_A': GRIDS[m][0], 'setting_B': GRIDS[m][1]} for m in METHODS]))

## 5. Classification, metrics, and CPU time

The same balanced logistic regression fits every two-coordinate training
map. A fixed 0.5 threshold gives a confusion matrix, accuracy, error,
sensitivity/recall, specificity, precision, F1, Cohen's kappa,
informedness, and markedness. ROC AUC and log loss use held-out
probabilities. CPU time covers embedding, test transform, classifier fit,
and prediction. UMAP compilation warmup is reported separately.

In [ ]:
def binary_metrics(y, prediction, probability):
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if precision + sensitivity else np.nan
    acc = (tp + tn) / len(y)
    prevalence = (tp + fn) / len(y)
    predicted = (tp + fp) / len(y)
    chance = prevalence * predicted + (1 - prevalence) * (1 - predicted)
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                accuracy=acc, error=1-acc, sensitivity=sensitivity,
                specificity=specificity, precision=precision, recall=sensitivity,
                f_measure=f1, auc_roc=roc_auc_score(y, probability),
                kappa=(acc-chance)/(1-chance),
                informedness=sensitivity+specificity-1,
                markedness=precision+npv-1,
                log_loss=log_loss(y, probability, labels=[0, 1]))

In [ ]:
def run(args):
    RESULTS.mkdir(exist_ok=True)
    (RESULTS / "raw").mkdir(exist_ok=True)
    selected_methods = METHODS if args.methods == "all" else [x.strip() for x in args.methods.split(",")]
    if any(method not in METHODS for method in selected_methods):
        raise ValueError(f"Unknown methods: {selected_methods}")
    warm_cpu = 0.0
    if "UMAP" in selected_methods:
        # Numba's first-call compilation is environment setup, not per-run fitting.
        warm = np.random.default_rng(7).normal(size=(50, 8)).astype(np.float32)
        warm_start = time.process_time()
        warm_model = umap.UMAP(n_neighbors=15, n_components=2, random_state=7, n_jobs=1)
        warm_model.fit(warm)
        warm_model.transform(warm[:5])
        warm_cpu = time.process_time() - warm_start
        print(f"UMAP JIT warmup CPU={warm_cpu:.2f}s", flush=True)
    rows, curves, traces = [], [], []
    if args.append:
        rows = pd.read_csv(RESULTS / "runs.csv").to_dict("records")
        curves = json.loads((RESULTS / "roc_curves.json").read_text(encoding="utf-8"))
        traces = json.loads((RESULTS / "optimization_traces.json").read_text(encoding="utf-8"))
        warm_cpu = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8")).get("umap_warmup_cpu_seconds", warm_cpu)
    for dataset in ["HAM10000", "Blood cell"]:
        x, y, groups, features = load_dataset(dataset)
        print(f"{dataset}: n={len(y)}, features={len(features)}, positives={int(y.sum())}", flush=True)
        for seed in range(args.seeds):
            xtr, xte, ytr, yte, tr, te = split_data(x, y, groups, 100 + seed,
                                                    args.train_n, args.test_n)
            for method in selected_methods:
                for setting, params in enumerate(GRIDS[method]):
                    if any(r["dataset"] == dataset and int(r["seed"]) == seed and
                           r["method"] == method and int(r["setting"]) == setting for r in rows):
                        continue
                    start_cpu = time.process_time()
                    start_wall = time.perf_counter()
                    with threadpool_limits(limits=1):
                        ztr, zte, trace = embed(method, params, xtr, xte, 100 + seed)
                        classifier = LogisticRegression(class_weight="balanced", max_iter=1000,
                                                        random_state=100 + seed)
                        classifier.fit(ztr, ytr)
                        prob = classifier.predict_proba(zte)[:, 1]
                        pred = (prob >= 0.5).astype(int)
                    cpu = time.process_time() - start_cpu
                    wall = time.perf_counter() - start_wall
                    metrics = binary_metrics(yte, pred, prob)
                    row = dict(dataset=dataset, seed=seed, method=method, setting=setting,
                               params=json.dumps(params, sort_keys=True), n_train=len(tr),
                               n_test=len(te), prevalence=float(yte.mean()), cpu_seconds=cpu,
                               wall_seconds=wall, **metrics)
                    rows.append(row)
                    fpr, tpr, thresholds = roc_curve(yte, prob)
                    curves.append(dict(dataset=dataset, seed=seed, method=method,
                                       setting=setting, fpr=fpr.tolist(), tpr=tpr.tolist()))
                    if trace is not None:
                        traces.append(dict(dataset=dataset, seed=seed, method=method,
                                           setting=setting, trace=trace))
                    np.savez_compressed(RESULTS / "raw" / f"{dataset.replace(' ','_')}_{seed}_{method.replace(' ','_')}_{setting}.npz",
                                        y=yte, pred=pred, prob=prob, test_index=te,
                                        embedding_test=zte)
                    print(f"{dataset} seed={seed} {method} {params}: AUC={metrics['auc_roc']:.3f} CPU={cpu:.2f}s", flush=True)
    pd.DataFrame(rows).to_csv(RESULTS / "runs.csv", index=False)
    (RESULTS / "roc_curves.json").write_text(json.dumps(curves), encoding="utf-8")
    (RESULTS / "optimization_traces.json").write_text(json.dumps(traces), encoding="utf-8")
    (RESULTS / "protocol.json").write_text(json.dumps({**vars(args), "all_methods": METHODS,
                                                         "umap_warmup_cpu_seconds": warm_cpu}, indent=2), encoding="utf-8")

In [ ]:
from argparse import Namespace
run(Namespace(seeds=SEEDS, train_n=TRAIN_N, test_n=TEST_N,
              methods="all", append=False))
runs = pd.read_csv(RESULTS / "runs.csv")
assert len(runs) == 2 * SEEDS * len(METHODS) * 2
display(runs[["dataset", "seed", "method", "setting", "accuracy",
              "auc_roc", "log_loss", "cpu_seconds"]].head(18))

## 6. Figures and statistical comparisons

The next cells generate accuracy, loss, ROC, confusion-matrix, CPU-time,
parameter-sensitivity, and SNE-objective plots. The **held-out classifier
log loss** is comparable across all methods; native embedding objectives
have different meanings and scales.

Paired t and Wilcoxon comparisons use shared split AUCs against PCA,
with Holm-adjusted t-test p values. Repeated-measures ANOVA tests the
method term. MANOVA is descriptive only because repeated splits violate
its independence assumption. McNemar z compares discordant first-split
predictions. Three overlapping splits give weak inferential power.

In [ ]:
OUT = RESULTS
FIG = RESULTS / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
COLORS = ["#153B50", "#23668A", "#4D9CB9", "#D17A42", "#AA4A3C", "#77619B", "#72944B", "#B28A32", "#576D77"]
COLORS += ['#A35F76', '#4B817A']

In [ ]:
def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / name, dpi=220, bbox_inches="tight")
    plt.close()

In [ ]:
def analyze_results():
    df = pd.read_csv(OUT / "runs.csv")
    primary = df[df.setting == 0].copy()
    cols = ["accuracy", "error", "sensitivity", "specificity", "precision",
            "recall", "f_measure", "auc_roc", "kappa", "informedness",
            "markedness", "log_loss", "cpu_seconds", "wall_seconds"]
    summary = primary.groupby(["dataset", "method"])[cols].agg(["mean", "std"])
    summary.columns = [f"{a}_{b}" for a,b in summary.columns]
    summary.reset_index().to_csv(OUT / "primary_summary.csv", index=False)
    df.groupby(["dataset", "method", "setting", "params"])[cols].mean().reset_index().to_csv(
        OUT / "parameter_comparison.csv", index=False)

    for dataset in primary.dataset.unique():
        part = primary[primary.dataset == dataset]
        ordered = part.groupby("method").accuracy.mean().reindex(METHODS)
        fig, ax = plt.subplots(figsize=(11, 4.5))
        bars = ax.bar(METHODS, ordered, color=COLORS)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Held-out accuracy")
        ax.set_title(f"{dataset}: accuracy by embedding")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.tick_params(axis="x", rotation=35)
        savefig(f"{dataset.replace(' ','_')}_accuracy.png")

        means = part.groupby("method").cpu_seconds.mean().reindex(METHODS)
        fig, ax = plt.subplots(figsize=(11, 4.5))
        bars = ax.bar(METHODS, means, color=COLORS)
        ax.set_ylabel("CPU seconds per fit and test")
        ax.set_title(f"{dataset}: process CPU time")
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
        ax.tick_params(axis="x", rotation=35)
        savefig(f"{dataset.replace(' ','_')}_cpu.png")

        curves = json.loads((OUT / "roc_curves.json").read_text(encoding="utf-8"))
        fig, ax = plt.subplots(figsize=(6.8, 5.4))
        for method, color in zip(METHODS, COLORS):
            items = [r for r in curves if r["dataset"] == dataset and r["seed"] == 0
                     and r["method"] == method and r["setting"] == 0]
            r = items[0]
            auc = float(part[(part.method == method) & (part.seed == 0)].auc_roc.iloc[0])
            ax.plot(r["fpr"], r["tpr"], label=f"{method} ({auc:.3f})", color=color)
        ax.plot([0, 1], [0, 1], ls="--", color="#999999")
        ax.set(xlabel="False positive rate", ylabel="Sensitivity", title=f"{dataset}: ROC on split 1")
        ax.legend(fontsize=8, loc="lower right")
        savefig(f"{dataset.replace(' ','_')}_roc.png")

        fig, axes = plt.subplots(3, 4, figsize=(12, 9.4))
        for ax, method in zip(axes.flat, METHODS):
            row = part[(part.method == method) & (part.seed == 0)].iloc[0]
            cm = np.array([[row.tn, row.fp], [row.fn, row.tp]], dtype=int)
            ax.imshow(cm, cmap="Blues")
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=13)
            ax.set(title=method, xticks=[0,1], yticks=[0,1], xlabel="Predicted", ylabel="True")
        for unused in list(axes.flat)[len(METHODS):]: unused.axis('off')
        fig.suptitle(f"{dataset}: held-out confusion matrices, split 1")
        savefig(f"{dataset.replace(' ','_')}_confusion.png")

        fig, ax = plt.subplots(figsize=(8.5, 4.5))
        for method, color in zip(METHODS, COLORS):
            sub = df[(df.dataset == dataset) & (df.method == method)]
            means = sub.groupby("setting").auc_roc.mean()
            ax.plot([0,1], means.values, marker="o", label=method, color=color)
        ax.set(xticks=[0,1], xticklabels=["Setting A", "Setting B"],
               ylabel="Mean ROC AUC", title=f"{dataset}: parameter sensitivity")
        ax.legend(ncol=3, fontsize=8)
        savefig(f"{dataset.replace(' ','_')}_parameters.png")

        loss = part.groupby("method").log_loss.mean().reindex(METHODS)
        fig, ax = plt.subplots(figsize=(11, 4.5))
        bars = ax.bar(METHODS, loss, color=COLORS)
        ax.set(ylabel="Held-out logistic log loss", title=f"{dataset}: common classifier loss")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.tick_params(axis="x", rotation=35)
        savefig(f"{dataset.replace(' ','_')}_loss.png")

    traces = json.loads((OUT / "optimization_traces.json").read_text(encoding="utf-8"))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, dataset in zip(axes, ["HAM10000", "Blood cell"]):
        trace = next(x["trace"] for x in traces if x["dataset"] == dataset and
                     x["seed"] == 0 and x["method"] == "SNE" and x["setting"] == 0)
        ax.plot([t[0] for t in trace], [t[1] for t in trace], color=COLORS[2])
        ax.set(xlabel="SNE optimization step", ylabel="Gaussian KL divergence", title=dataset)
    savefig("sne_objective.png")

    # This is intentionally an applicability figure: no lesion localization or
    # multiple candidate detections exist in the available tables.
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.axis("off")
    ax.text(0.5, .7, "FROC AUC cannot be estimated from these labels", ha="center",
            fontsize=17, weight="bold", color="#153B50")
    ax.text(0.5, .35, "Required: lesion locations and scored candidate detections per image.\n"
            "Available: one binary label and one score per record.", ha="center", fontsize=12)
    savefig("froc_applicability.png")

    stat_rows = []
    for dataset in primary.dataset.unique():
        part = primary[primary.dataset == dataset]
        pivot = part.pivot(index="seed", columns="method", values="auc_roc")[METHODS]
        base = pivot["PCA"].to_numpy()
        pvals = []
        pending = []
        for method in METHODS[1:]:
            other = pivot[method].to_numpy()
            t = stats.ttest_rel(other, base)
            try:
                w = stats.wilcoxon(other - base)
                wp = float(w.pvalue)
            except ValueError:
                wp = np.nan
            pending.append((method, float(np.mean(other-base)), float(t.statistic), float(t.pvalue), wp))
            pvals.append(float(t.pvalue))
        adjusted = multipletests(pvals, method="holm")[1]
        for (method, difference, tstat, tp, wp), adj in zip(pending, adjusted):
            stat_rows.append(dict(dataset=dataset, comparison=f"{method} vs PCA",
                                  metric="AUC ROC", mean_difference=difference,
                                  paired_t=tstat, paired_t_p=tp, paired_t_holm_p=float(adj),
                                  wilcoxon_p=wp))
    pd.DataFrame(stat_rows).to_csv(OUT / "paired_tests.csv", index=False)

    # Repeated-measures ANOVA respects the shared splits. The MANOVA below is
    # requested as a descriptive sensitivity analysis, not confirmatory evidence.
    a = primary.copy()
    a["subject"] = a.dataset + "_" + a.seed.astype(str)
    try:
        anova = AnovaRM(a, depvar="auc_roc", subject="subject", within=["method"]).fit()
        (OUT / "anova.txt").write_text(str(anova), encoding="utf-8")
    except Exception as exc:
        (OUT / "anova.txt").write_text(f"Not estimable: {exc}", encoding="utf-8")
    try:
        manova = MANOVA.from_formula("auc_roc + log_loss ~ method + dataset", data=a).mv_test()
        (OUT / "manova.txt").write_text(str(manova), encoding="utf-8")
    except Exception as exc:
        (OUT / "manova.txt").write_text(f"Not estimable: {exc}", encoding="utf-8")

    # First split, discordant paired predictions: McNemar's large-sample z.
    zrows = []
    for dataset in primary.dataset.unique():
        stem = dataset.replace(" ", "_")
        base = np.load(OUT / "raw" / f"{stem}_0_PCA_0.npz")
        for method in METHODS[1:]:
            other = np.load(OUT / "raw" / f"{stem}_0_{method.replace(' ','_')}_0.npz")
            y = base["y"]
            b = np.sum((base["pred"] == y) & (other["pred"] != y))
            c = np.sum((base["pred"] != y) & (other["pred"] == y))
            z = (c-b) / np.sqrt(b+c) if b+c else np.nan
            p = 2*stats.norm.sf(abs(z)) if np.isfinite(z) else np.nan
            zrows.append(dict(dataset=dataset, comparison=f"{method} vs PCA",
                              discordant_pca_only=int(b), discordant_other_only=int(c),
                              mcnemar_z=z, two_sided_p=p))
    pd.DataFrame(zrows).to_csv(OUT / "mcnemar_z.csv", index=False)
    print(summary.round(3).to_string())

In [ ]:
analyze_results()
summary = pd.read_csv(RESULTS / "primary_summary.csv")
display(summary[["dataset", "method", "accuracy_mean", "error_mean",
                 "sensitivity_mean", "specificity_mean", "precision_mean",
                 "f_measure_mean", "auc_roc_mean", "kappa_mean",
                 "informedness_mean", "markedness_mean", "log_loss_mean",
                 "cpu_seconds_mean"]].round(3))

In [ ]:
for dataset in ("HAM10000", "Blood cell"):
    part = summary[summary.dataset == dataset].set_index("method").reindex(METHODS)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    bars = ax.bar(METHODS, part.auc_roc_mean, color=COLORS)
    ax.set(ylabel="Mean held-out ROC AUC", ylim=(0, 1),
           title=f"{dataset}: ranking quality across 11 embeddings")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(FIG / f"{dataset.replace(' ', '_')}_auc.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

In [ ]:
for dataset in ("HAM10000", "Blood cell"):
    stem = dataset.replace(" ", "_")
    print(f"\n{dataset}: AUC, accuracy, loss, ROC, and confusion matrices")
    for kind in ("auc", "accuracy", "loss", "roc", "confusion"):
        display(Image(filename=str(FIG / f"{stem}_{kind}.png")))
    print(f"{dataset}: CPU time and parameter sensitivity")
    for kind in ("cpu", "parameters"):
        display(Image(filename=str(FIG / f"{stem}_{kind}.png")))
display(Image(filename=str(FIG / "sne_objective.png")))

### Where the classes lie in each two-dimensional map

Blue points are negative records and orange points are positive records
on the first held-out split under setting A. Axes differ by method and
must not be compared numerically across panels. Overlap suggests a simple
linear classifier will struggle; apparent separation still needs
held-out AUC, precision, and calibration checks.

In [ ]:
for dataset in ("HAM10000", "Blood cell"):
    fig, axes = plt.subplots(3, 4, figsize=(15, 10))
    stem = dataset.replace(" ", "_")
    for ax, method in zip(axes.flat, METHODS):
        path = RESULTS / "raw" / f"{stem}_0_{method.replace(' ', '_')}_0.npz"
        with np.load(path) as result:
            z, y = result["embedding_test"], result["y"]
        ax.scatter(z[y == 0, 0], z[y == 0, 1], s=13, alpha=.6,
                   color="#23668A", label="negative")
        ax.scatter(z[y == 1, 0], z[y == 1, 1], s=19, alpha=.8,
                   color="#D17A42", label="positive")
        auc = runs[(runs.dataset == dataset) & (runs.seed == 0) &
                   (runs.method == method) & (runs.setting == 0)].auc_roc.iloc[0]
        ax.set_title(f"{method} | AUC {auc:.3f}", fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in list(axes.flat)[len(METHODS):]: ax.axis("off")
    axes.flat[0].legend(fontsize=7)
    fig.suptitle(f"{dataset}: class-colored held-out embeddings", fontsize=17)
    fig.tight_layout()
    image_path = FIG / f"{stem}_class_embeddings.png"
    fig.savefig(image_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(image_path)))

In [ ]:
hybrids = summary[summary.method.isin(["PCA", "t-SNE", "PCA + t-SNE",
                                             "UMAP", "PCA + UMAP", "LLE"])]
display(hybrids[["dataset", "method", "auc_roc_mean", "accuracy_mean",
                 "cpu_seconds_mean"]].sort_values(["dataset", "auc_roc_mean"],
                                                   ascending=[True, False]).round(3))

## 7. Interpretation and limitations

The best AUC method need not have the best accuracy at 0.5 or the lowest
loss. With rare melanoma, accuracy can hide low positive precision. AUCs
near 0.6 show only modest discrimination in this pilot. Proposed geometric
reasons for a method's result are hypotheses rather than proven causes.

In [ ]:
winners = []
for dataset in ("HAM10000", "Blood cell"):
    part = summary[summary.dataset == dataset]
    auc = part.loc[part.auc_roc_mean.idxmax()]
    acc = part.loc[part.accuracy_mean.idxmax()]
    cpu = part.loc[part.cpu_seconds_mean.idxmin()]
    winners.append({"dataset": dataset, "best_AUC": auc.method,
                    "AUC": auc.auc_roc_mean, "best_accuracy": acc.method,
                    "accuracy": acc.accuracy_mean, "fastest": cpu.method,
                    "CPU_seconds": cpu.cpu_seconds_mean})
display(pd.DataFrame(winners).round(3))
for dataset in ("HAM10000", "Blood cell"):
    part = summary[summary.dataset == dataset].set_index("method")
    novel = part.loc["PCA + UMAP"]
    other = part.drop(index="PCA + UMAP").sort_values("auc_roc_mean", ascending=False).iloc[0]
    print(f"{dataset}: PCA + UMAP AUC {novel.auc_roc_mean:.3f}, "
          f"best other method {other.name} {other.auc_roc_mean:.3f}; "
          f"PCA + UMAP CPU {novel.cpu_seconds_mean:.2f}s.")
print("Small AUC gaps over three overlapping splits do not establish an improvement.")
print("LLE's accuracy should also be read with its AUC, sensitivity, and precision.")
display(pd.read_csv(RESULTS / "paired_tests.csv").round(4))
display(pd.read_csv(RESULTS / "mcnemar_z.csv").round(4))
print((RESULTS / "anova.txt").read_text(encoding="utf-8"))
print("MANOVA is descriptive, not a confirmatory significance test.")
print((RESULTS / "manova.txt").read_text(encoding="utf-8"))

### FROC AUC cannot be estimated from these tables

FROC needs scored candidate locations per image to plot sensitivity
against false positives **per image**. These files contain one binary
label and one score per record. Calling ROC a FROC would invent a
localization result. The figure below explains the missing requirement.

In [ ]:
display(Image(filename=str(FIG / 'froc_applicability.png')))
print(f'All results: {RESULTS}')

## 8. Next study and data sources

The `results/` directory contains per-run metrics, raw held-out
predictions, plots, parameter tables, and statistical outputs. A stronger
study needs original lesion images or validated features, patient-level
IDs for the blood-cell cohort, external cohorts, nested tuning, and
uncertainty intervals. Detection annotations are needed if FROC is a
target.

[HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000)
and [Blood Cell Anomaly Detection 2025](https://www.kaggle.com/datasets/alitaqishah/blood-cell-anomaly-detection-2025)
retain their respective source terms. Cite them separately.